<a href="https://colab.research.google.com/github/Chikka-Pradhayani/ABTalks-60-Days-AI-Challenge/blob/main/Day-24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install fastapi uvicorn langchain langchain-openai openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 3.5 MB/s eta 0:00:00


In [4]:
import os
from typing import Dict, List, Optional
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [7]:
class ConversationHistory:
    def __init__(self):
        self.messages = []

    def append(self, role, content):
        self.messages.append({
            "role": role,
            "content": content
        })

    def get_context(self, last_five_turns=5):
        # Keep only the latest 5 conversation turns
        return self.messages[-(last_five_turns * 2):]

    def clear(self):
        self.messages.clear()

    def turn_count(self):
        return sum(
            1 for message in self.messages
            if message["role"] == "user"
        )

In [8]:
history = ConversationHistory()

history.append("user", "My project is EcoTrack.")
history.append("assistant", "That sounds interesting!")

history.append("user", "It predicts household electricity usage.")
history.append("assistant", "Got it.")

print(history.get_context())

[{'role': 'user', 'content': 'My project is EcoTrack.'}, {'role': 'assistant', 'content': 'That sounds interesting!'}, {'role': 'user', 'content': 'It predicts household electricity usage.'}, {'role': 'assistant', 'content': 'Got it.'}]


In [9]:
history = ConversationHistory()

for i in range(1, 8):
    history.append("user", f"User message {i}")
    history.append("assistant", f"Assistant response {i}")

print("Total turns:", history.turn_count())
print("\nLast 5 turns:")

for message in history.get_context(5):
    print(message)

Total turns: 7

Last 5 turns:
{'role': 'user', 'content': 'User message 3'}
{'role': 'assistant', 'content': 'Assistant response 3'}
{'role': 'user', 'content': 'User message 4'}
{'role': 'assistant', 'content': 'Assistant response 4'}
{'role': 'user', 'content': 'User message 5'}
{'role': 'assistant', 'content': 'Assistant response 5'}
{'role': 'user', 'content': 'User message 6'}
{'role': 'assistant', 'content': 'Assistant response 6'}
{'role': 'user', 'content': 'User message 7'}
{'role': 'assistant', 'content': 'Assistant response 7'}


In [10]:
history.clear()

print("Messages after clear:", history.messages)
print("Turns after clear:", history.turn_count())

Messages after clear: []
Turns after clear: 0


In [11]:
sessions = {}

def get_session(session_id):
    if session_id not in sessions:
        sessions[session_id] = ConversationHistory()
    return sessions[session_id]

In [12]:
# Session 1
user1 = get_session("user_001")
user1.append("user", "My project is called EcoTrack.")
user1.append("assistant", "Got it. Your project is EcoTrack.")

# Session 2
user2 = get_session("user_002")
user2.append("user", "My project is called HealthAI.")
user2.append("assistant", "Got it. Your project is HealthAI.")

print("User 1:")
print(user1.get_context())

print("\nUser 2:")
print(user2.get_context())

User 1:
[{'role': 'user', 'content': 'My project is called EcoTrack.'}, {'role': 'assistant', 'content': 'Got it. Your project is EcoTrack.'}]

User 2:
[{'role': 'user', 'content': 'My project is called HealthAI.'}, {'role': 'assistant', 'content': 'Got it. Your project is HealthAI.'}]


In [13]:
user1.append("user", "It predicts electricity usage.")

print("User 1:")
print(user1.get_context())

print("\nUser 2:")
print(user2.get_context())

User 1:
[{'role': 'user', 'content': 'My project is called EcoTrack.'}, {'role': 'assistant', 'content': 'Got it. Your project is EcoTrack.'}, {'role': 'user', 'content': 'It predicts electricity usage.'}]

User 2:
[{'role': 'user', 'content': 'My project is called HealthAI.'}, {'role': 'assistant', 'content': 'Got it. Your project is HealthAI.'}]


In [14]:
history = ConversationHistory()

messages = [
    "My project is called EcoTrack.",
    "It predicts household electricity usage.",
    "What does it predict?",
    "What technologies would be suitable for it?",
    "Can you give me a short description of it?"
]

for message in messages:
    history.append("user", message)

    # Simulated assistant response
    response = "Response based on the available conversation memory."
    history.append("assistant", response)

print("Conversation:")
for msg in history.get_context(5):
    print(msg["role"], ":", msg["content"])

Conversation:
user : My project is called EcoTrack.
assistant : Response based on the available conversation memory.
user : It predicts household electricity usage.
assistant : Response based on the available conversation memory.
user : What does it predict?
assistant : Response based on the available conversation memory.
user : What technologies would be suitable for it?
assistant : Response based on the available conversation memory.
user : Can you give me a short description of it?
assistant : Response based on the available conversation memory.


In [15]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Day 24 Conversation Memory")


class ChatRequest(BaseModel):
    message: str
    session_id: str = "default"


class ChatResponse(BaseModel):
    session_id: str
    answer: str
    turns_stored: int

In [16]:
@app.post("/chat")
def chat(request: ChatRequest):

    # Get the correct session
    history = get_session(request.session_id)

    # Get previous 5 turns
    context = history.get_context(5)

    # Store user's message
    history.append("user", request.message)

    # Temporary response
    answer = (
        f"I received your message: '{request.message}'. "
        f"I can access {len(context)} previous messages from this session."
    )

    # Store assistant response
    history.append("assistant", answer)

    return {
        "session_id": request.session_id,
        "answer": answer,
        "turns_stored": history.turn_count()
    }

In [17]:
print(app.routes)

[Route(path='/openapi.json', name='openapi', methods=['GET', 'HEAD']), Route(path='/docs', name='swagger_ui_html', methods=['GET', 'HEAD']), Route(path='/docs/oauth2-redirect', name='swagger_ui_redirect', methods=['GET', 'HEAD']), Route(path='/redoc', name='redoc_html', methods=['GET', 'HEAD']), APIRoute(path='/chat', name='chat', methods=['POST'])]


In [18]:
def truncate_memory(history):
    # Only truncate when conversation exceeds 10 turns
    if history.turn_count() <= 10:
        return

    # Get all messages
    messages = history.messages

    # First 10 messages = oldest 5 turns
    oldest_messages = messages[:10]

    # Create a simple summary for now
    summary = "Summary of the oldest 5 turns: "

    for msg in oldest_messages:
        summary += f"{msg['role']}: {msg['content']} | "

    # Keep messages after the oldest 5 turns
    remaining_messages = messages[10:]

    # Replace old messages with summary
    history.messages = [
        {
            "role": "system",
            "content": summary
        }
    ] + remaining_messages

In [19]:
history = ConversationHistory()

for i in range(1, 13):
    history.append("user", f"User message {i}")
    history.append("assistant", f"Assistant response {i}")

print("Before truncation:")
print("Total turns:", history.turn_count())

Before truncation:
Total turns: 12


In [20]:
truncate_memory(history)

print("\nAfter truncation:")
print("Messages:")

for msg in history.messages:
    print(msg["role"], ":", msg["content"])


After truncation:
Messages:
system : Summary of the oldest 5 turns: user: User message 1 | assistant: Assistant response 1 | user: User message 2 | assistant: Assistant response 2 | user: User message 3 | assistant: Assistant response 3 | user: User message 4 | assistant: Assistant response 4 | user: User message 5 | assistant: Assistant response 5 | 
user : User message 6
assistant : Assistant response 6
user : User message 7
assistant : Assistant response 7
user : User message 8
assistant : Assistant response 8
user : User message 9
assistant : Assistant response 9
user : User message 10
assistant : Assistant response 10
user : User message 11
assistant : Assistant response 11
user : User message 12
assistant : Assistant response 12


In [21]:
@app.post("/chat")
def chat(request: ChatRequest):

    # Get session
    history = get_session(request.session_id)

    # Truncate old memory if needed
    truncate_memory(history)

    # Get latest 5 turns
    context = history.get_context(5)

    # Store user message
    history.append("user", request.message)

    # Temporary response
    answer = (
        f"Received: '{request.message}'. "
        f"I have {len(context)} previous messages in memory."
    )

    # Store assistant response
    history.append("assistant", answer)

    return {
        "session_id": request.session_id,
        "answer": answer,
        "turns_stored": history.turn_count(),
        "memory_messages": len(history.messages)
    }

In [22]:
test_messages = [
    "My project is called EcoTrack. It predicts household electricity usage.",
    "I built it using Python and scikit-learn.",
    "What does it predict?",
    "Why would someone use it?",
    "Can you give me a short resume description for it?"
]

print("5 test messages loaded successfully.")

5 test messages loaded successfully.


In [23]:
def run_with_memory(messages):
    history = ConversationHistory()
    outputs = []

    for message in messages:

        context = history.get_context(5)

        # Simulated AI response using the available memory
        if "What does it predict?" in message:
            answer = "EcoTrack predicts household electricity usage."

        elif "Why would someone use it?" in message:
            answer = "Someone could use EcoTrack to monitor and reduce energy waste."

        elif "resume description" in message:
            answer = (
                "Developed EcoTrack using Python and scikit-learn "
                "to predict household electricity usage and support energy efficiency."
            )

        else:
            answer = "Got it. I will remember that."

        history.append("user", message)
        history.append("assistant", answer)

        outputs.append(answer)

    return outputs


memory_on = run_with_memory(test_messages)

for i, answer in enumerate(memory_on, 1):
    print(f"Turn {i}: {answer}")

Turn 1: Got it. I will remember that.
Turn 2: Got it. I will remember that.
Turn 3: EcoTrack predicts household electricity usage.
Turn 4: Someone could use EcoTrack to monitor and reduce energy waste.
Turn 5: Developed EcoTrack using Python and scikit-learn to predict household electricity usage and support energy efficiency.


In [24]:
def run_without_memory(messages):
    outputs = []

    for message in messages:

        # AI only receives the current message
        if "What does it predict?" in message:
            answer = "I don't have enough context to know what 'it' refers to."

        elif "Why would someone use it?" in message:
            answer = "I don't know what 'it' refers to."

        elif "resume description" in message:
            answer = "Please provide the project name and details."

        else:
            answer = "Got it."

        outputs.append(answer)

    return outputs


memory_off = run_without_memory(test_messages)

for i, answer in enumerate(memory_off, 1):
    print(f"Turn {i}: {answer}")

Turn 1: Got it.
Turn 2: Got it.
Turn 3: I don't have enough context to know what 'it' refers to.
Turn 4: I don't know what 'it' refers to.
Turn 5: Please provide the project name and details.


In [25]:
print("========== MEMORY ON ==========")

for i, answer in enumerate(memory_on, 1):
    print(f"{i}. {answer}")

print("\n========== MEMORY OFF ==========")

for i, answer in enumerate(memory_off, 1):
    print(f"{i}. {answer}")

========== MEMORY ON ==========
1. Got it. I will remember that.
2. Got it. I will remember that.
3. EcoTrack predicts household electricity usage.
4. Someone could use EcoTrack to monitor and reduce energy waste.
5. Developed EcoTrack using Python and scikit-learn to predict household electricity usage and support energy efficiency.

========== MEMORY OFF ==========
1. Got it.
2. Got it.
3. I don't have enough context to know what 'it' refers to.
4. I don't know what 'it' refers to.
5. Please provide the project name and details.


In [26]:
# Day 24 Token Cost Analysis

daily_users = 1000
turns_per_session = 10

# Assumed average tokens
user_tokens_per_message = 75
assistant_tokens_per_message = 150

tokens_per_turn = user_tokens_per_message + assistant_tokens_per_message

# With a 5-turn memory window:
# Turn 1 → 0 previous turns
# Turn 2 → 1
# Turn 3 → 2
# ...
# Turn 6-10 → 5 each

previous_context_turns = sum(
    min(i, 5) for i in range(turns_per_session)
)

extra_tokens_per_session = (
    previous_context_turns * tokens_per_turn
)

extra_tokens_per_day = (
    daily_users * extra_tokens_per_session
)

extra_tokens_per_month = (
    extra_tokens_per_day * 30
)

# Example GPT-4o-mini input price
price_per_million_tokens = 0.15

monthly_cost = (
    extra_tokens_per_month / 1_000_000
) * price_per_million_tokens


print("========== TOKEN COST ANALYSIS ==========")
print("Average tokens per turn:", tokens_per_turn)
print("Previous context turns/session:", previous_context_turns)
print("Extra tokens/session:", extra_tokens_per_session)
print("Extra tokens/day:", extra_tokens_per_day)
print("Extra tokens/month:", extra_tokens_per_month)
print("Estimated monthly memory cost: $", round(monthly_cost, 2))

========== TOKEN COST ANALYSIS ==========
Average tokens per turn: 225
Previous context turns/session: 35
Extra tokens/session: 7875
Extra tokens/day: 7875000
Extra tokens/month: 236250000
Estimated monthly memory cost: $ 35.44


In [27]:
print("""
DAY 24 — CONCLUSION

A session-aware conversation memory system was implemented using
ConversationHistory and session_id based storage.

The system:
1. Stores user and assistant messages.
2. Maintains separate memory for each session.
3. Injects only the latest five conversation turns.
4. Truncates old conversations after ten turns.
5. Compares conversations with and without memory.
6. Calculates the additional token cost of maintaining memory.

Memory improves follow-up questions because previous conversation
context is available to the assistant. The trade-off is additional
input-token usage and therefore higher API cost.
""")


DAY 24 — CONCLUSION

A session-aware conversation memory system was implemented using
ConversationHistory and session_id based storage.

The system:
1. Stores user and assistant messages.
2. Maintains separate memory for each session.
3. Injects only the latest five conversation turns.
4. Truncates old conversations after ten turns.
5. Compares conversations with and without memory.
6. Calculates the additional token cost of maintaining memory.

Memory improves follow-up questions because previous conversation
context is available to the assistant. The trade-off is additional
input-token usage and therefore higher API cost.

